## **Messwerte**

In [69]:
import numpy as np

R = 8.314462 #ideale Gaskonstante in Si-Einheiten

T = np.array([20.8, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56]) + 273.15 #Temperatur in Kelvin
T_Fehler = 0.2

eta = np.array([60.2, 57.8, 54.9, 51.4, 48.3, 45.1, 42.5, 39.8, 37.3, 34.7, 33.0, 30.9, 29.1, 27.5, 25.9, 24.5, 23.0, 21.9, 20.7])*10**-3 #Viskosität in Pa*s
eta_Fehler = 0.2e-3


## **Lineare Regression**

In [70]:
import scipy
from Skripte.Fehlerfortpflanzung import Gaußfehler
import sympy
from IPython.display import display


n = sympy.symbols("eta")
expr = sympy.log(n)
display(expr)

logEta = np.log(eta)
print(logEta)
logEta_Fehler = np.ones(len(logEta), dtype=float)

for x in range(0, len(logEta)): #Fehler für log(eta) berechnen
    logEta_Fehler[x] = Gaußfehler(expr, np.array([n]), np.array([logEta[x]]), np.array([eta_Fehler]))

def f(x, a, c):#Fit Funktion
    return a*x + c

coefficients, pcov = scipy.optimize.curve_fit(f, 1/T, logEta, sigma= logEta_Fehler, absolute_sigma= True)

k = coefficients[0] #Steigung der Geraden
k_Fehler = np.sqrt(pcov[0,0])
print(f"\nSteigung: k = {k} +/- {k_Fehler}")

c = coefficients[1] #Y-Achsen Abschnitt der Geraden
c_Fehler = np.sqrt(pcov[1,1])
print(f"\nY-Achsenabschnitt: a = {c} +/- {c_Fehler}")


log(eta)

[-2.81008293 -2.8507665  -2.90224193 -2.96811711 -3.03032372 -3.09887303
 -3.1582512  -3.22388837 -3.28876195 -3.36101559 -3.41124772 -3.4769991
 -3.5370171  -3.59356927 -3.65351231 -3.70908216 -3.77226106 -3.82126864
 -3.87762158]

Steigung: k = 2971.64114514131 +/- 0.12382145542384027

Y-Achsenabschnitt: a = -12.905328725656373 +/- 0.00039586369431399913


## **Plot**

In [71]:
import matplotlib.pyplot as plt
import matplotlib


matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'arial',
    'text.usetex': True,
    'pgf.rcfonts': False,
})


fig, ax1 = plt.subplots(1,1, figsize = (5,5))
ax1.errorbar(1/T, logEta, yerr = logEta_Fehler, fmt = "o", label = "Datapoints", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")

xdata = np.linspace(np.min(1/T)*0.99, np.max(1/T)*1.01)
ax1.plot(xdata, k*xdata + c, color = "blue", linestyle = "--", label = "Linear fit")

ax1.set_xlabel("$\\frac{1}{T}~~[\\frac{1}{K}]$")
ax1.set_ylabel("$\\ln(\\eta)~~[\\ln(~Pa\\cdot s)]$")
ax1.grid(True)
ax1.set_title("Arrhenius Plot")
ax1.legend()

plt.savefig("Thrixtropie.pgf")

## **Aktivierungsenergie berechnen**

In [72]:

E_a = k*R #Formel für Aktivierungsenergie aus Steigung der Geraden

kp = sympy.symbols("k")
expr2 = -kp*R
E_a_Fehler = Gaußfehler(expr2, np.array([kp]), np.array([k]), np.array([k_Fehler]))

print(f"\nAktivierungsenergie: E_a = {E_a} +/- {E_a_Fehler} J")




Aktivierungsenergie: E_a = 24707.59737891391 +/- 1.029508785906214 J


## **Flächenbestimmung**

In [73]:
import scipy
import pandas as pd

data = pd.read_excel("Thrixtropie.xlsx", usecols = "C,D")

dataArray = data.to_numpy()


gamma = dataArray[0:267,1]
tau = dataArray[0:267,0]

fig, ax2 = plt.subplots(1,1, figsize = (6,6))

ax2.plot(gamma[0:119], tau[0:119], color = "black")
ax2.plot(gamma[::-1][:120], tau[::-1][:120], color = "black")
ax2.set_xlabel("$\\dot{\\gamma}$")
ax2.set_ylabel("$\\tau$")
ax2.grid(True)
ax2.set_title("")


Fläche_Gesamt = scipy.integrate.simpson(tau[0:119], gamma[0:119])

Fläche_unten = scipy.integrate.simpson(tau[::-1][:120], gamma[::-1][:120])

TI = (Fläche_Gesamt - Fläche_unten) / Fläche_Gesamt
print(TI*100)


7.854305437833248
